In [5]:
import os
import numpy as np
import pandas as pd

FLOOR_SUMMARY = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_summary_approach_1_avoid_step_floor.csv"
ELBOW_SUMMARY = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_approach_1_elbow_step.csv"
OUT_DIR       = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"

BATCH_SIZES = [64, 1024, 60000]

for tag, path in [("floor", FLOOR_SUMMARY), ("elbow", ELBOW_SUMMARY)]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[{tag}] summary not found:\n  {path}\n"
            f"Run that method's compute notebook first, or fix the path above."
        )

floor_df = pd.read_csv(FLOOR_SUMMARY); floor_df.columns = floor_df.columns.str.strip()
elbow_df = pd.read_csv(ELBOW_SUMMARY); elbow_df.columns = elbow_df.columns.str.strip()
print("floor summary:", FLOOR_SUMMARY)
print("elbow summary:", ELBOW_SUMMARY)
print(f"Loaded {len(floor_df)} / {len(elbow_df)} pruning rows.")

floor summary: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_floor\ipa_summary_approach_1_avoid_step_floor.csv
elbow summary: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_approach_1_elbow_step.csv
Loaded 19 / 19 pruning rows.


In [1]:
!pip install kneed

In [2]:
print("""
=== Hyperparameter Comparison ===

SHARED by both methods (step detection + empirical floor asymptote):
  BN_STEP_MIN = 100   -- Minimum BN before checking for step artifacts.
  STEP_THRESH = 0.01  -- |CE[i] - CE[i-1]| must exceed this to trigger a step cutoff.
  TAIL_N      = 50    -- Empirical floor A = mean of the last TAIL_N data points.
                         A is pinned (not fitted) to reflect the true data plateau.

FLOOR method  (approach_1_avoid_step_floor):
  THRESHOLD = 0.90    -- CE_L = CE_o - 0.90 * (CE_o - A)
                         Sets the learning threshold 90% of the way from CE_o down to A.
                         BN_learned = first batch where averaged CE crosses this threshold.
                         IPA = |CE_o - CE_L| / BN_learned.
                         The choice of 0.90 is ARBITRARY — no principled basis.

ELBOW method  (approach_1_elbow_step, this work):
  KNEEDLE_S = 1.0     -- Kneedle sensitivity parameter (Satopää et al., 2011).
                         Controls how conservative the knee detection is:
                           S = 0  -> least conservative, equivalent to argmax(D)
                           S = 1  -> default; filters out weak peaks in D
                           S > 1  -> more conservative, knee found later in the curve
                         Unlike the floor threshold, S is a shape parameter, not a
                         CE-level parameter — it controls which peak of the D curve
                         counts as the knee, not where on the CE axis to threshold.

  NO CE-LEVEL THRESHOLD. Knee found automatically:
    1. Normalize BN to [0,1]: x_norm = (BN - BN_min) / (BN_max - BN_min)
    2. Normalize CE to [0,1]: y_norm = (CE - CE_min) / (CE_max - CE_min)
    3. D[i] = x_norm[i] + y_norm[i] - 1  (distance above the start-to-end diagonal)
    4. knee = first peak of D exceeding S * amplitude(D)
    5. BN_learned = knee BN,  CE_learned = CE at the knee data point
    6. IPA = |CE_o - CE_learned| / BN_learned
""")


=== Hyperparameter Comparison ===

SHARED by both methods (step detection + empirical floor asymptote):
  BN_STEP_MIN = 100   -- Minimum BN before checking for step artifacts.
  STEP_THRESH = 0.01  -- |CE[i] - CE[i-1]| must exceed this to trigger a step cutoff.
  TAIL_N      = 50    -- Empirical floor A = mean of the last TAIL_N data points.
                         A is pinned (not fitted) to reflect the true data plateau.

FLOOR method  (approach_1_avoid_step_floor):
  THRESHOLD = 0.90    -- CE_L = CE_o - 0.90 * (CE_o - A)
                         Sets the learning threshold 90% of the way from CE_o down to A.
                         BN_learned = first batch where averaged CE crosses this threshold.
                         IPA = |CE_o - CE_L| / BN_learned.
                         The choice of 0.90 is ARBITRARY — no principled basis.

ELBOW method  (approach_1_elbow_step, this work):
  KNEEDLE_S = 1.0     -- Kneedle sensitivity parameter (Satopää et al., 2011).
                  

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

fig, axes = plt.subplots(1, len(BATCH_SIZES), figsize=(6 * len(BATCH_SIZES), 5), sharex=True)
if len(BATCH_SIZES) == 1:
    axes = [axes]

for ax, bs in zip(axes, BATCH_SIZES):
    col = f"IPA_Avg_{bs}"
    f_sub = floor_df.dropna(subset=[col]) if col in floor_df.columns else floor_df.iloc[0:0]
    e_sub = elbow_df.dropna(subset=[col]) if col in elbow_df.columns else elbow_df.iloc[0:0]
    ax.plot(f_sub["P%"].values, f_sub[col].values, "o--", color="#999999", ms=5, lw=1.6,
            label="floor (threshold=0.90)")
    ax.plot(e_sub["P%"].values, e_sub[col].values, "o-",  color=BS_COLOR.get(bs, "#1f77b4"), ms=5, lw=2,
            label="elbow (kneedle S=1.0)")
    ax.set_title(f"BS={bs}")
    ax.set_xlabel("Pruning Percentage (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False, fontsize=10)
axes[0].set_ylabel("IPA")
fig.suptitle("IPA vs Pruning  —  floor threshold=0.90 (dashed grey) vs Kneedle S=1.0 (solid)",
             fontsize=12)

out_png = os.path.join(OUT_DIR, "ipa_compare_floor_vs_elbow.png")
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

NameError: name 'BATCH_SIZES' is not defined

In [ ]:
# Numeric comparison: diff = elbow - floor (positive = elbow gives higher IPA)
merged = floor_df[["P%"]].copy()
for bs in BATCH_SIZES:
    col = f"IPA_Avg_{bs}"
    if col in floor_df.columns and col in elbow_df.columns:
        merged[f"floor_{bs}"] = floor_df[col].values
        merged[f"elbow_{bs}"] = elbow_df[col].values
        merged[f"diff_{bs}"]  = elbow_df[col].values - floor_df[col].values
print(merged.to_string(index=False))